In [4]:
import logging
from datetime import datetime
from pyspark.sql.functions import col

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

notebook_name = "Pipeline_Monitor"
run_start = datetime.utcnow()
logger.info(f"[START] {notebook_name} — {run_start.strftime('%Y-%m-%d %H:%M:%S')} UTC")

#  Check today's run log 
today = datetime.utcnow().date()

run_log = spark.read.format("delta").load("Tables/dbo/pipeline_run_log")

todays_runs = run_log.filter(
    col("run_start").cast("date") == str(today)
).orderBy("run_start", ascending=False)

print("=== AutoTrust Pipeline Run Summary ===")
print(f"Date: {today}")
print(f"Total notebooks run: {todays_runs.count()}")
todays_runs.select(
    "notebook_name",
    "run_start",
    "duration_secs",
    "status"
).show(10, truncate=False)

#  Flag any failures 
failures = todays_runs.filter(col("status") != "SUCCESS")
failure_count = failures.count()

if failure_count > 0:
    print(f"\n[ALERT] {failure_count} notebook(s) had issues today:")
    failures.select("notebook_name", "status", "run_start").show(truncate=False)
else:
    print("\n[OK] All notebooks completed successfully today")

#  Check Gold tables were written today 
gold_tables = [
    "gold_dim_make",
    "gold_dim_vehicle",
    "gold_dim_failure_category",
    "gold_fact_vehicle_reliability"
]

print("\n=== Gold Table Row Counts (post-run) ===")
for table in gold_tables:
    try:
        count = spark.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
        print(f"{table}: {count:,} rows")
    except Exception as e:
        print(f"{table}: ERROR — {e}")

StatementMeta(, 1b96a4c1-4614-4a47-9f7d-5fce3b170e58, 6, Finished, Available, Finished, False)

INFO:__main__:[START] Pipeline_Monitor — 2026-04-24 03:36:07 UTC


=== AutoTrust Pipeline Run Summary ===
Date: 2026-04-24
Total notebooks run: 6
+----------------+--------------------------+-------------+-------+
|notebook_name   |run_start                 |duration_secs|status |
+----------------+--------------------------+-------------+-------+
|Pipeline_Monitor|2026-04-24 03:33:48.390813|19           |SUCCESS|
|Gold            |2026-04-24 03:19:37.17674 |94           |SUCCESS|
|Gold            |2026-04-24 03:10:08.721127|262          |SUCCESS|
|Silver          |2026-04-24 03:07:20.991761|85           |SUCCESS|
|Bronze          |2026-04-24 03:03:00.139947|66           |SUCCESS|
|Bronze_ingestion|2026-04-24 02:32:21.73143 |1345         |SUCCESS|
+----------------+--------------------------+-------------+-------+


[OK] All notebooks completed successfully today

=== Gold Table Row Counts (post-run) ===
gold_dim_make: 27 rows
gold_dim_vehicle: 77 rows
gold_dim_failure_category: 275 rows
gold_fact_vehicle_reliability: 1,314 rows


In [5]:
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

run_end = datetime.utcnow()
duration = (run_end - run_start).seconds

log_schema = StructType([
    StructField("notebook_name",  StringType(),    True),
    StructField("run_start",      TimestampType(), True),
    StructField("run_end",        TimestampType(), True),
    StructField("duration_secs",  IntegerType(),   True),
    StructField("status",         StringType(),    True),
])

log_row = [Row(
    notebook_name = notebook_name,
    run_start     = run_start,
    run_end       = run_end,
    duration_secs = duration,
    status        = "SUCCESS"
)]

log_df = spark.createDataFrame(log_row, schema=log_schema)

log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("pipeline_run_log")

print(f"[DONE] {notebook_name} — {duration}s")

StatementMeta(, 1b96a4c1-4614-4a47-9f7d-5fce3b170e58, 7, Finished, Available, Finished, False)

[DONE] Pipeline_Monitor — 72s
